In [0]:
from delta.tables import DeltaTable
from pyspark.sql.functions import col, to_timestamp, explode, row_number
from pyspark.sql.window import Window

# 1. Setup Variables
catalog = "maritime_ais"
bronze_schema = "maritime_bronze"
silver_schema = "maritime_silver"
checkpoint_base = "abfss://maritime-lake@maritimepipeline.dfs.core.windows.net/checkpoints/silver"

# 2. Define Upsert Logic (Spark Connect safe + dedup)
def upsert_to_silver(microBatchDF, batchId, table_name, merge_condition, dedup_keys=None):
    try:
        _spark = microBatchDF.sparkSession

        if dedup_keys:
            w = Window.partitionBy(*dedup_keys).orderBy(col(dedup_keys[0]))
            microBatchDF = (
                microBatchDF.withColumn("_rn", row_number().over(w))
                .filter(col("_rn") == 1)
                .drop("_rn")
            )

        if not _spark.catalog.tableExists(table_name):
            microBatchDF.write.format("delta").mode("overwrite").saveAsTable(table_name)
            return

        delta_table = DeltaTable.forName(_spark, table_name)
        delta_table.alias("target").merge(
            microBatchDF.alias("source"),
            merge_condition
        ).whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()

    except Exception as e:
        print(f"CRITICAL ERROR in batch {batchId}: {str(e)}")
        raise e

# 3. Process Port Calls
print("Processing Silver Port Calls...")
df_bronze_ports = spark.readStream.table(f"{catalog}.{bronze_schema}.port_calls")

df_exploded_ports = df_bronze_ports.select(explode(col("portCalls")).alias("pc"))

df_silver_ports = df_exploded_ports.select(
    col("pc.portCallId").cast("string").alias("port_call_id"),
    col("pc.mmsi").cast("string").alias("mmsi"),
    to_timestamp(col("pc.portCallTimestamp")).alias("call_timestamp"),
    col("pc.portToVisit").alias("destination_port_code"),
    col("pc.prevPort").alias("previous_port_code"),
    col("pc.vesselName").alias("vessel_name"),
    to_timestamp(col("pc.portAreaDetails").getItem(0)["eta"]).alias("eta"),
    col("pc.portAreaDetails").getItem(0)["portAreaName"].alias("port_name")
)

ports_table = f"{catalog}.{silver_schema}.port_calls"

# Port calls are keyed by their unique ID
merge_condition = "target.port_call_id = source.port_call_id"

query_ports = (
    df_silver_ports.writeStream
    .foreachBatch(lambda df, epoch_id: upsert_to_silver(
        df, epoch_id, ports_table, merge_condition, dedup_keys=["port_call_id"]
    ))
    .option("checkpointLocation", f"{checkpoint_base}/port_calls")
    .trigger(availableNow=True)
    .start()
)
query_ports.processAllAvailable()
print("Completed Port Calls Processing.")